# Воспроизводимость: sklearn Pipeline

##Подготовка к построению Pipeline:

Предобработка признаков и модель, реализованные в `01_training.ipynb`, объединяются в единый sklearn Pipeline с последующим сохранением и загрузкой.

Для обеспечения независимого запуска повторно выполняются загрузка данных из S3, подготовка данных, разделение выборок и настройка предобработки:

In [1]:
!pip install boto3

In [2]:
import boto3
import pandas as pd
import os
from google.colab import userdata

os.environ["KEY_ID"] = userdata.get("KEY_ID")
os.environ["ACCESS_KEY"] = userdata.get("ACCESS_KEY")

KEY_ID = os.environ["KEY_ID"]
ACCESS_KEY = os.environ["ACCESS_KEY"]

S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/listings.csv"

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

In [3]:
def read_s3_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(response["Body"])

In [4]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


In [5]:
df = df_s3.drop_duplicates()

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X = df[["latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds", "minimum_nights", "neighbourhood_cleansed", "property_type", "room_type"]]
y = df["price"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [10]:
numeric_columns = ["latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds", "minimum_nights"]
category_columns = ["neighbourhood_cleansed", "property_type", "room_type"]

In [11]:
numeric_pipeline = Pipeline([('fill', SimpleImputer(strategy='median'))])

In [12]:
categorical_pipeline = Pipeline([('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [13]:
preprocessor = ColumnTransformer([('numbers', numeric_pipeline, numeric_columns), ('category', categorical_pipeline, category_columns)], remainder='drop')

##Создание и обучение Pipeline:

Предобработка признаков и модель `RandomForestRegressor`
объединяются в единый **sklearn Pipeline**, состоящий из 2 этапов:

- prepare (заполнение пропусков и кодирование категориальных признаков)
- model (модель регрессии)

Все обучаемые преобразования и модель настраиваются на тренировочных данных.

In [14]:
from sklearn.ensemble import RandomForestRegressor

In [15]:
final_pipeline = Pipeline([('prepare', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42))])
final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('prepare',
                 ColumnTransformer(transformers=[('numbers',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['latitude', 'longitude',
                                                   'accommodates', 'bathrooms',
                                                   'bedrooms', 'beds',
                                                   'minimum_nights']),
                                                 ('category',
                                                  Pipeline(steps=[('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['neighbourhood_cleansed',
                                                   'property_type',
                                                   'room_type'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [16]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [17]:
y_pred = final_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 3))

MAE: 66.64
RMSE: 246.73
R2: 0.494


##Сохранение Pipeline:

Обученный **Pipeline** сохраняется в файл `model.pkl` с помощью библиотеки `pickle`. Сохраняется вся цепочка целиком.

In [18]:
import pickle
from pathlib import Path

In [19]:
artifact_dir = Path('artifacts')
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / 'model.pkl'

In [20]:
with model_path.open('wb') as file:
    pickle.dump(final_pipeline, file)

##Загрузка Pipeline и получение предсказаний:

Сохранённый **Pipeline** загружается с помощью `pickle.load()`. Предсказания выполняются на исходных признаках без повторного обучения модели и отдельной предобработки данных.

In [22]:
with model_path.open('rb') as file:
    loaded_model = pickle.load(file)

In [23]:
y_pred = loaded_model.predict(X_test)

In [24]:
import numpy as np

print('Предсказания совпадают:', np.array_equal(final_pipeline.predict(X_test), y_pred))

Предсказания совпадают: True


In [25]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 3))

MAE: 66.64
RMSE: 246.73
R2: 0.494


После загрузки получены предсказания без повторного обучения.
Значения метрик совпадают с результатами `01_training.ipynb`, качество модели сохранено.